In [10]:
from transformers import VitsModel, AutoTokenizer
import torch
import numpy as np
import scipy.io.wavfile as wavfile
import os


In [11]:
MODEL_NAME = "procit001/nepali_male_v1"
SAVE_PATH = "./saved_nepali_tts_model"
OUTPUT_PATH = "final_nepali_audio.wav"

device = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
print("Downloading/loading model from Hugging Face...")

model = VitsModel.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Model and tokenizer saved successfully!")

Downloading/loading model from Hugging Face...
Model and tokenizer saved successfully!


In [13]:
print("Loading saved model...")

model = VitsModel.from_pretrained(SAVE_PATH).to(device)
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

model.eval()

print("Saved model loaded successfully!")

Loading saved model...
Saved model loaded successfully!


In [14]:
def split_nepali_sentences(text):
    sentences = text.replace("।", "।|").split("|")
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

In [15]:
def generate_sentence_audio(sentence):
    inputs = tokenizer(sentence, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model(**inputs).waveform

    audio = output.squeeze().cpu().numpy()

    return audio


In [16]:
def generate_full_audio(text, output_path=OUTPUT_PATH):
    sentences = split_nepali_sentences(text)

    print(f"Total sentences found: {len(sentences)}")

    all_audio = []

    # small silence between sentences
    silence_duration = 0.3
    silence = np.zeros(int(model.config.sampling_rate * silence_duration))

    for i, sentence in enumerate(sentences):
        print(f"Generating sentence {i+1}/{len(sentences)}:")
        print(sentence)

        audio = generate_sentence_audio(sentence)

        all_audio.append(audio)
        all_audio.append(silence)

    final_audio = np.concatenate(all_audio)

    # Normalize audio
    max_val = np.max(np.abs(final_audio))
    if max_val > 0:
        final_audio = final_audio / max_val

    # Convert to int16 WAV format
    final_audio = (final_audio * 32767).astype(np.int16)

    wavfile.write(
        output_path,
        rate=model.config.sampling_rate,
        data=final_audio
    )

    print(f"Final audio saved as: {output_path}")
    return output_path

In [17]:
text = """
नेपाल एक सुन्दर, विविधतायुक्त र प्राकृतिक सम्पदाले भरिएको देश हो। यहाँ हिमाल, पहाड र तराई गरी तीन मुख्य भौगोलिक क्षेत्र छन्। उत्तरतिर सेता हिमालहरू मुस्कुराइरहेका देखिन्छन् भने दक्षिणतिर हरियाली खेतबारी र समथर तराई फैलिएको छ। नेपाल सानो देश भए पनि यहाँको भाषा, संस्कृति, परम्परा, रहनसहन र जीवनशैली निकै विविध छन्। यही विविधता नै नेपालको वास्तविक पहिचान हो।

नेपालका गाउँहरूमा बिहान सबेरै मानिसहरू उठ्छन्। कसैले गाईभैँसीलाई घाँस दिन्छन्, कसैले खेतबारीमा काम गर्न जान्छन्, अनि विद्यार्थीहरू किताब बोकेर विद्यालयतिर लाग्छन्। गाउँको जीवन सरल, मेहनती र प्रकृतिसँग नजिक हुन्छ। बिहानको चिसो हावा, चराहरूको आवाज, टाढाबाट सुनिने खोलाको कलकल ध्वनि र खेतमा देखिने हरियालीले मन शान्त बनाउँछ। शहरको व्यस्त जीवनभन्दा गाउँको वातावरण फरक हुन्छ, तर दुवै ठाउँका आफ्नै विशेषता छन्।

शहरमा मानिसहरूको जीवन छिटो गतिमा चल्छ। कार्यालय जाने मानिस, विद्यालय जाने विद्यार्थी, व्यापार गर्ने व्यापारी, सडकमा गुड्ने सवारी साधन र बजारको भीडले शहर सधैँ सक्रिय देखिन्छ। प्रविधिको विकाससँगै शहरहरूमा इन्टरनेट, मोबाइल, कम्प्युटर र आधुनिक सेवाहरूको प्रयोग बढ्दै गएको छ। मानिसहरूले अनलाइन पढाइ, अनलाइन व्यापार, डिजिटल भुक्तानी र सामाजिक सञ्जालको प्रयोग दिनानुदिन बढाइरहेका छन्। यसले जीवन सजिलो बनाएको छ, तर समय व्यवस्थापन र मानसिक शान्ति जस्ता कुरामा चुनौती पनि थपिएको छ।

शिक्षा कुनै पनि व्यक्तिको जीवन परिवर्तन गर्ने मुख्य माध्यम हो। शिक्षाले मानिसलाई सोच्न, बुझ्न, प्रश्न गर्न र सही निर्णय लिन सिकाउँछ। शिक्षित व्यक्ति आफ्नो मात्र होइन, परिवार, समाज र देशको विकासमा पनि योगदान दिन सक्छ। नेपालमा शिक्षाको पहुँच पहिलेभन्दा बढेको छ। धेरै बालबालिका विद्यालय जान थालेका छन्। तर अझै पनि दुर्गम क्षेत्रका केही बालबालिकाले गुणस्तरीय शिक्षा पाउन कठिनाइ भोगिरहेका छन्। त्यसैले शिक्षा सबैका लागि समान र सहज बनाउनु आजको ठूलो आवश्यकता हो।

कृषि नेपालको प्रमुख आधार हो। धेरै नेपालीहरू खेतीपाती र पशुपालनमा निर्भर छन्। धान, मकै, गहुँ, कोदो, आलु, तरकारी र फलफूल यहाँका मुख्य कृषि उत्पादनहरू हुन्। किसानहरूले वर्षभरि मेहनत गर्छन्। कहिले पानीको अभाव हुन्छ, कहिले मल र बीउको समस्या आउँछ, कहिले रोग र किराले बाली नोक्सान गर्छ। तर पनि किसानहरूले हार मान्दैनन्। आधुनिक कृषि प्रविधि, सिँचाइ सुविधा, राम्रो बीउ, रोग व्यवस्थापन र बजार पहुँच राम्रो भएमा नेपालको कृषि अझ उत्पादनशील बन्न सक्छ।

स्वास्थ्य पनि मानिसको जीवनको महत्वपूर्ण पक्ष हो। स्वस्थ शरीर र स्वस्थ मन भए मात्र मानिसले राम्रोसँग काम गर्न सक्छ। सफा पानी पिउनु, पोषिलो खाना खानु, नियमित व्यायाम गर्नु, पर्याप्त निद्रा लिनु र सरसफाइमा ध्यान दिनु स्वास्थ्यका आधारभूत कुरा हुन्। गाउँ तथा शहर दुवै ठाउँमा स्वास्थ्य चेतना बढाउन आवश्यक छ। साना रोगलाई बेवास्ता गर्दा पछि ठूलो समस्या हुन सक्छ। त्यसैले समयमै स्वास्थ्य परीक्षण गर्ने बानी बसाल्नु राम्रो हुन्छ।

पर्यावरण संरक्षण आजको अत्यन्त महत्वपूर्ण विषय हो। वनजंगल, नदी, ताल, हिमाल, जनावर र चराचुरुङ्गी हाम्रो प्राकृतिक सम्पत्ति हुन्। यी सम्पत्तिको संरक्षण नगरेमा भविष्यका पुस्ताले ठूलो समस्या भोग्नुपर्ने हुन्छ। जथाभावी रूख काट्नु, नदीमा फोहोर फाल्नु, प्लास्टिकको अत्यधिक प्रयोग गर्नु र प्रदूषण फैलाउनु वातावरणका लागि हानिकारक छ। हरेक नागरिकले कम्तीमा एउटा रूख रोप्ने, फोहोर छुट्याएर व्यवस्थापन गर्ने, पानी बचत गर्ने र प्रकृतिलाई माया गर्ने बानी विकास गर्नुपर्छ।

नेपालको संस्कृति संसारमै विशेष छ। यहाँ विभिन्न जातजाति, भाषा, धर्म र परम्पराका मानिसहरू मिलेर बसेका छन्। दशैँ, तिहार, छठ, ल्होसार, ईद, बुद्ध जयन्ती, होली र अन्य धेरै चाडपर्वहरू खुशी र एकताको प्रतीक हुन्। चाडपर्वले परिवार र समाजलाई नजिक ल्याउँछ। मानिसहरूले आफ्ना दुःख, तनाव र व्यस्ततालाई केही समयका लागि बिर्सेर रमाइलो गर्छन्। तर चाडपर्व मनाउँदा सरलता, सद्भाव र जिम्मेवारी पनि आवश्यक हुन्छ।

प्रविधिले नेपालको भविष्यमा ठूलो भूमिका खेल्न सक्छ। आजका युवाहरू कम्प्युटर, कृत्रिम बुद्धिमत्ता, रोबोटिक्स, वेब विकास, मोबाइल एप, डेटा विज्ञान र साइबर सुरक्षाजस्ता क्षेत्रमा रुचि देखाइरहेका छन्। सही सीप, अवसर र मार्गदर्शन पाएमा नेपाली युवाहरू विश्व बजारमा प्रतिस्पर्धा गर्न सक्छन्। प्रविधिको प्रयोग शिक्षा, स्वास्थ्य, कृषि, पर्यटन र सरकारी सेवामा प्रभावकारी रूपमा गर्न सके देशको विकास छिटो हुन सक्छ। तर प्रविधिको प्रयोग जिम्मेवारीपूर्वक गर्नुपर्छ। गलत सूचना फैलाउने, अरूको गोपनीयता तोड्ने र समयको दुरुपयोग गर्ने बानीबाट टाढा रहनुपर्छ।

पर्यटन नेपालको अर्को महत्वपूर्ण क्षेत्र हो। सगरमाथा, अन्नपूर्ण, लुम्बिनी, पोखरा, चितवन, रारा ताल, मुस्ताङ, जनकपुर र काठमाडौं उपत्यका जस्ता स्थानहरूले देशलाई विश्वमा चिनाएका छन्। विदेशी पर्यटकहरू यहाँको प्राकृतिक सौन्दर्य, संस्कृति, साहसिक यात्रा र आतिथ्यताबाट प्रभावित हुन्छन्। पर्यटनले रोजगारी सिर्जना गर्छ, स्थानीय उत्पादनको बजार बढाउँछ र देशको अर्थतन्त्रमा सहयोग पुर्याउँछ। तर पर्यटन विकास गर्दा वातावरण, स्थानीय संस्कृति र समुदायको सम्मान पनि गर्नुपर्छ।

युवाहरू देशका आधार हुन्। उनीहरूमा ऊर्जा, नयाँ सोच, साहस र सिर्जनशीलता हुन्छ। यदि युवाहरूलाई राम्रो शिक्षा, सीपमूलक तालिम, रोजगारी र उद्यमशीलताको अवसर दिइयो भने देशको मुहार फेरिन सक्छ। युवाहरूले विदेश जाने मात्र होइन, आफ्नै देशमा सम्भावना खोज्ने सोच पनि विकास गर्नुपर्छ। तर त्यसका लागि राज्य, समाज र परिवारले सहयोगी वातावरण बनाउनुपर्छ। इमानदारी, अनुशासन, मेहनत र धैर्य सफल जीवनका मुख्य आधार हुन्।

समाजमा सकारात्मक परिवर्तन ल्याउन सबैको भूमिका हुन्छ। केवल सरकारको भर परेर मात्र विकास सम्भव हुँदैन। नागरिकले पनि नियम पालना गर्ने, सार्वजनिक सम्पत्तिको संरक्षण गर्ने, अरूलाई सम्मान गर्ने, कर तिर्ने, भ्रष्टाचारको विरोध गर्ने र सहयोगी भावना राख्ने गर्नुपर्छ। साना साना राम्रा कामहरूले ठूलो परिवर्तन ल्याउन सक्छन्। बाटोमा फोहोर नफाल्नु, लाइनमा पालो कुर्नु, वृद्धवृद्धा र अशक्तलाई सहयोग गर्नु, सत्य बोल्नु र जिम्मेवारी पूरा गर्नु पनि सभ्य समाजका संकेत हुन्।

अन्त्यमा, नेपाल सम्भावनाले भरिएको देश हो। यहाँ प्राकृतिक स्रोत छन्, मेहनती मानिस छन्, प्रतिभाशाली युवा छन् र समृद्ध संस्कृति छ। चुनौतीहरू अवश्य छन्, तर चुनौतीसँगै अवसर पनि छन्। सबैले आफ्नो ठाउँबाट इमानदारीपूर्वक काम गर्ने हो भने नेपाललाई शिक्षित, स्वस्थ, समृद्ध र आत्मनिर्भर देश बनाउन सकिन्छ। देशको विकास कुनै एक दिनमा हुँदैन। त्यसका लागि निरन्तर प्रयास, धैर्य, एकता र सकारात्मक सोच आवश्यक हुन्छ। हामी सबैले आफ्नो कर्तव्य बुझेर अघि बढ्यौँ भने नेपालको भविष्य उज्यालो हुनेछ।
"""

generate_full_audio(text)

Total sentences found: 75
Generating sentence 1/75:
नेपाल एक सुन्दर, विविधतायुक्त र प्राकृतिक सम्पदाले भरिएको देश हो।
Generating sentence 2/75:
यहाँ हिमाल, पहाड र तराई गरी तीन मुख्य भौगोलिक क्षेत्र छन्।
Generating sentence 3/75:
उत्तरतिर सेता हिमालहरू मुस्कुराइरहेका देखिन्छन् भने दक्षिणतिर हरियाली खेतबारी र समथर तराई फैलिएको छ।
Generating sentence 4/75:
नेपाल सानो देश भए पनि यहाँको भाषा, संस्कृति, परम्परा, रहनसहन र जीवनशैली निकै विविध छन्।
Generating sentence 5/75:
यही विविधता नै नेपालको वास्तविक पहिचान हो।
Generating sentence 6/75:
नेपालका गाउँहरूमा बिहान सबेरै मानिसहरू उठ्छन्।
Generating sentence 7/75:
कसैले गाईभैँसीलाई घाँस दिन्छन्, कसैले खेतबारीमा काम गर्न जान्छन्, अनि विद्यार्थीहरू किताब बोकेर विद्यालयतिर लाग्छन्।
Generating sentence 8/75:
गाउँको जीवन सरल, मेहनती र प्रकृतिसँग नजिक हुन्छ।
Generating sentence 9/75:
बिहानको चिसो हावा, चराहरूको आवाज, टाढाबाट सुनिने खोलाको कलकल ध्वनि र खेतमा देखिने हरियालीले मन शान्त बनाउँछ।
Generating sentence 10/75:
शहरको व्यस्त जीवनभन्दा गाउँको वाता

'final_nepali_audio.wav'